# Data Cleaning


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
import hashlib
import re 
from sklearn.model_selection import TimeSeriesSplit
raw_data_path= '../Data/Raw Data/'
clean_data_path= '../Data/Clean Data/'
raw_data_file=raw_data_path+'raw_data.json'
df=pd.read_json(raw_data_file)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1718613 entries, 0 to 1718612
Data columns (total 47 columns):
 #   Column                Dtype  
---  ------                -----  
 0   transaction_id        str    
 1   procedure_id          int64  
 2   trans_group_id        int64  
 3   trans_group_ar        str    
 4   trans_group_en        str    
 5   procedure_name_ar     str    
 6   procedure_name_en     str    
 7   instance_date         str    
 8   property_type_id      int64  
 9   property_type_ar      str    
 10  property_type_en      str    
 11  property_sub_type_id  float64
 12  property_sub_type_ar  str    
 13  property_sub_type_en  str    
 14  property_usage_ar     str    
 15  property_usage_en     str    
 16  reg_type_id           int64  
 17  reg_type_ar           str    
 18  reg_type_en           str    
 19  area_id               int64  
 20  area_name_ar          str    
 21  area_name_en          str    
 22  building_name_ar      str    
 23  building_name_en  

In [2]:
# Rename columns for convenience
column_names={
    'TRANSACTION_NUMBER':'transaction_num',
    'INSTANCE_DATE':'instance_date',
    'GROUP_EN':'group',
    'PROCEDURE_EN':'procedure',
    'IS_OFFPLAN_EN':'is_offplan',
    'IS_FREE_HOLD_EN':'is_free_hold',
    'USAGE_EN':'usage',
    'AREA_EN':'area',
    'PROP_TYPE_EN':'prop_type',
    'PROP_SB_TYPE_EN':'prop_sb_type',
    'TRANS_VALUE':'trans_value',
    'PROCEDURE_AREA':'procedure_area',
    'ACTUAL_AREA':'actual_area',
    'ROOMS_EN':'rooms',
    'PARKING':'parking',
    'NEAREST_METRO_EN':'nearest_metro',
    'NEAREST_MALL_EN':'nearest_mall',
    'NEAREST_LANDMARK_EN':'nearest_landmark',
    'TOTAL_BUYER':'total_buyer',
    'TOTAL_SELLER':'total_seller',
    'MASTER_PROJECT_EN':'master_project',
    'PROJECT_EN':'project'
}
df.rename(columns=column_names,inplace=True)

#Hash transaction number
df['transaction_num']=df['transaction_num'].apply(lambda x:hashlib.sha256(str(x).encode()).hexdigest())
#Drop requried columns
df=df.drop(columns=['total_buyer','total_seller','master_project'])

In [3]:
df = df[df['group'] == 'Sales'].copy()
df = df[df['prop_type'] == 'Unit'].copy()
df = df.drop_duplicates().reset_index(drop=True)

In [4]:
#Turn date datatype to datetime
df['instance_date']=pd.to_datetime(df['instance_date'])
df['date_year']= df['instance_date'].dt.year
#Months turned into sin and cos vectors
df['month_sin']=np.sin(2 * np.pi*df['instance_date'].dt.month/12.0)
df['month_cos']=np.cos(2*np.pi*df['instance_date'].dt.month/12.0)

In [5]:
#Calculate the baseline price per square foot
df['price_per_sqft']=df['trans_value']/df['actual_area']

#Sort chronologically within each community to properly calculate rolling windows
df=df.sort_values(by=['area','instance_date']).reset_index(drop=True)

#Set the date as the index for time based rolling math
df=df.set_index('instance_date')

#Generate the rolling 30 day average price per square foot for each specific area
df['community_30d_avg_price']=df.groupby('area')['price_per_sqft'].transform(lambda x:x.shift(1).rolling('30D').mean())

df['_offplan_flag'] = (df['is_offplan'] == 'Off-Plan').astype(int)
df['off_plan_vol_ratio'] = (df.groupby('area')['_offplan_flag'].transform(lambda x: x.shift(1).rolling('30D', min_periods=1).mean()))
df = df.drop(columns=['_offplan_flag'])

df.reset_index(inplace=True)
df['community_30d_avg_price'] = df.groupby('area')['community_30d_avg_price'].transform(lambda x: x.fillna(x.mean()))
df['off_plan_vol_ratio'] = df.groupby('area')['off_plan_vol_ratio'].transform(lambda x: x.fillna(x.mean()))
# Areas with only one transaction → no group mean → fill with global mean
df['community_30d_avg_price'] = df['community_30d_avg_price'].fillna(df['price_per_sqft'].mean())
df['off_plan_vol_ratio'] = df['off_plan_vol_ratio'].fillna(0.5)

In [6]:
#Clean up rooms values
df['rooms']=df['rooms'].astype(str).str.extract(r'(\d+)').fillna(0).astype(int)

#Clean parking by finding first number found. if not found return nan
def clean_parking(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    nums = re.findall(r'\d+', s)
    if not nums:
        return 1 if s else np.nan
    if re.match(r'^[A-Za-z]+[-\s]\d+$', s):
        return 1
    total = sum(int(n) for n in nums)
    return min(total, 10)
df['parking'] = df['parking'].apply(clean_parking)

In [7]:
df['is_offplan'] = df['is_offplan'].map({'Off-Plan': 1, 'Ready': 0}).fillna(0).astype(int)
df['is_free_hold'] = df['is_free_hold'].map({'Free Hold': 1, 'Non Free Hold': 0}).fillna(0).astype(int)
df['usage'] = df['usage'].map({'Residential': 1, 'Commercial': 0}).fillna(1).astype(int)

In [8]:
#Fill categorical columns with Unknown
cat_cols_with_na=['prop_sb_type','nearest_metro','nearest_mall','nearest_landmark','project']
df[cat_cols_with_na]=df[cat_cols_with_na].fillna('Unknown')
#Fill numerical columns with median
num_cols_with_na=['rooms','parking']
for col in num_cols_with_na:
    df[col]=df[col].fillna(df[col].median())

In [ ]:
#Remove rows with negative or zero area/value
df = df[df['trans_value']>0]
df = df[df['actual_area']>0]
df = df[df['procedure_area']>0]

#Remove extreme top 1% outliers for numerical features
for col in ['trans_value','actual_area','procedure_area']:
    q01=df[col].quantile(0.01)
    q99=df[col].quantile(0.99)
    df = df[(df[col] >= q01) & (df[col] <= q99)]
df['price_per_sqft'] = df['trans_value'] / df['actual_area']
pps_low, pps_high = df['price_per_sqft'].quantile([0.001, 0.999])
df = df[(df['price_per_sqft'] >= pps_low) & (df['price_per_sqft'] <= pps_high)].reset_index(drop=True)
print(f"After outlier removal: {len(df):,} rows")

After outlier removal: 55,743 rows


In [ ]:
df = df.sort_values('instance_date').reset_index(drop=True)
df = df.drop(columns=['procedure_area'])
proc_counts = df['procedure'].value_counts()
rare_procs = proc_counts[proc_counts < len(df) * 0.005].index
df['procedure'] = df['procedure'].where(~df['procedure'].isin(rare_procs), 'Other')

def oof_target_encode(df, col, target, n_splits=5, smoothing=10):
    global_mean = df[target].mean()
    encoded = pd.Series(index=df.index, dtype=float)
    # df must be sorted by instance_date BEFORE this is called
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for train_idx, val_idx in tscv.split(df):
        agg = df.iloc[train_idx].groupby(col)[target].agg(['mean', 'count'])
        smoothed = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)
        encoded.iloc[val_idx] = df.iloc[val_idx][col].map(smoothed).fillna(global_mean)
    # First fold of TimeSeriesSplit has no training data → those rows stay NaN
    encoded = encoded.fillna(global_mean)
    return encoded

df['area'] = oof_target_encode(df, 'area', 'price_per_sqft')

for col in ['project', 'prop_sb_type', 'nearest_metro', 'nearest_mall', 'nearest_landmark']:
    df[col] = oof_target_encode(df, col, 'price_per_sqft')

df = df.drop(columns=['group', 'prop_type'])
df = pd.get_dummies(df, columns=['procedure'], drop_first=True)

In [ ]:
#Drop columns that have the same value across all cells as to remvoe noise
df = df.drop(columns=['instance_date','price_per_sqft'])
cons_cols=[x for x in df.columns if df[x].nunique()<=1]
if cons_cols:
    df=df.drop(columns=cons_cols)
df.to_csv(clean_data_path + 'cleaned_data.csv', index=False)

In [ ]:
df.to_csv(clean_data_path + 'cleaned_data.csv', index=False)

In [13]:
df=pd.read_csv(clean_data_path+'cleaned_data.csv')
df.head()

,transaction_num,is_offplan,is_free_hold,usage,area,prop_sb_type,trans_value,actual_area,rooms,parking,...,off_plan_vol_ratio,procedure_Delayed Sell,procedure_Development Registration,procedure_Development Registration Pre-Registration,procedure_Lease to Own Registration,procedure_Other,procedure_Sale,procedure_Sale On Payment Plan,procedure_Sell - Pre registration,procedure_Sell Development
0,8d597dea9d5b17e4864bb967d682dfe0c2c4e3e3bf59de...,1,1,1,22871.458721,20973.249073,1099573.00,72.41,1,1.0,...,0.980223,False,False,False,False,False,False,False,True,False
1,e7e3e032c67f5804fe5338439686914e420f9e5f37810f...,1,1,1,15154.818304,20973.249073,720622.22,38.69,0,1.0,...,0.924072,False,False,False,False,False,False,False,True,False
2,fb56be070ace6ba2fb5deb176cff3d3224b7eb748d0b28...,1,1,1,15154.818304,20973.249073,716711.12,38.69,0,1.0,...,1.000000,False,False,False,False,False,False,False,True,False
3,c3be6f190979ac104182ae13025c4b5887e4f61a5cc911...,1,1,1,15125.835821,20930.544206,713777.79,38.54,0,1.0,...,1.000000,False,False,False,False,False,False,False,True,False
4,6dc70d3cc57802594a77f74c9d77359a246c5efea87ff7...,1,1,1,26330.685543,20966.018766,2097828.00,77.64,1,1.0,...,0.934171,False,False,False,False,False,False,False,True,False
